<!-- TASK_A_REWRITTEN_ANALYSIS -->
# Dyck Syn-to-Rea: Task A Length and Noise

这个 notebook 的问题是：Transformer 在带 noise 的 Dyck-1 synthetic sequence 上，到底是学到了 Dyck 计数机制，还是只学到了一些局部 readout / shortcut？

阅读顺序如下。

1. **实验设置与主结果。** 先看 6 个 Transformer settings 的 behavior、oracle forced/free split、以及 hidden-state count probe。这里的核心结论是：普通 settings 接近 stochastic generator 的 oracle ceiling；`tiny_extreme_long/b20` 是真正失败的稀疏监督 regime。

2. **固定 seq_len=2000 的 sparsity sweep。** 再只改变 bracket token 数量，定位 b20 到 b100 之间的转变。这一段说明：forced open/close 规则其实较早能在 bracket 子空间中读出，但 full-vocab 输出需要足够 bracket mass。

3. **基础 probes 和 ablations。** 然后检查 height/left/right 是否线性可读、是否跨 setting 稳定、以及 height direction 是否直接接到 output head。这里的核心区分是：可线性读出不等于被模型用于输出。

4. **CountScope-style online patching。** 这一段做 source/target/patched forward pass。`local_interchange` 测当前位置 readout 是否能被 hidden state 改写；`future_continued` 测较早 patch 是否能影响后续 forced decision。结果支持 late hidden 对局部 open/close readout 有因果作用，但不支持稳定的 continued counter-state transfer。

5. **Axis/span patching 与 sparse failure diagnostics。** 最后把 readout 方向、Dyck-target CE/acc、forced/free split、bracket-only 指标和 bracket mass 放在一起看。这一段是目前最重要的机制解释：模型确实表示 count/left/right；但 next-bracket 输出主要接在与 output head 对齐的 forced-decision direction 上。低 density 失败主要来自 bracket token 在 full vocab 中输给 noise，而不是单纯 open/close 子空间失效。

因此，看这个 notebook 时不要只盯 overall accuracy。更可靠的读法是同时看：`forced/free split`、`Dyck-target full-vocab CE/acc`、`bracket-only CE/acc`、`bracket mass`，以及 patching/ablation 是否显示某个方向真的被 output path 使用。

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

CONFIG_TEMPLATE = ROOT / 'configs' / 'dyck_counter_length_noise_transformer.yaml'
GENERATED_CONFIG_DIR = ROOT / 'configs' / 'generated_task_a'
RESULTS_ROOT = ROOT / 'results'

WRITE_CONFIGS = False
RUN_TRAINING = False
DEVICE = "cuda"  
NUM_EXAMPLES = 4096
EXTRACT_BATCH_SIZE = 512
PROBE_MAX_ROWS = 30000
PROBE_MAX_CLASSES = 200

ROOT

## Experiment Settings

所有设置都使用同一个 3-layer Transformer、seed=0、final checkpoint hidden states，并抽取 all layers / all positions。变化的只有 Dyck 总长度、序列长度、噪声密度和 noise vocabulary。

当前 Dyck sampler 是 stochastic Markov-style balanced path：当 height=0、open 数用完、或剩余步数必须全 close 时，下一步由规则强制决定；其余 free step 近似 50/50 open-vs-close。这个生成机制非常重要，因为它给 next-token accuracy 设置了一个 Bayes-optimal 上限。


In [ ]:
TASK_A_GROUPS = [
    dict(name='tiny_extreme_long', dyck_pairs=10, total_length=20, seq_len=2000, repeat_prob=0.01, num_noise_tokens=64, seeds=[0], batch_size=4),
    dict(name='clean_short', dyck_pairs=24, total_length=48, seq_len=48, repeat_prob=1.0, num_noise_tokens=4, seeds=[0], batch_size=128),
    dict(name='noisy_short', dyck_pairs=24, total_length=48, seq_len=120, repeat_prob=0.5, num_noise_tokens=16, seeds=[0], batch_size=128),
    dict(name='sparse_medium', dyck_pairs=100, total_length=200, seq_len=400, repeat_prob=0.25, num_noise_tokens=16, seeds=[0], batch_size=128),
    dict(name='sparse_long', dyck_pairs=200, total_length=400, seq_len=1000, repeat_prob=0.25, num_noise_tokens=16, seeds=[0], batch_size=16),
    dict(name='extreme_long', dyck_pairs=200, total_length=400, seq_len=2000, repeat_prob=0.1, num_noise_tokens=64, seeds=[0], batch_size=4),
]

pd.DataFrame(TASK_A_GROUPS)


In [ ]:
def make_group_config(group: dict) -> dict:
    cfg = yaml.safe_load(CONFIG_TEMPLATE.read_text())
    cfg['experiment']['name'] = f"dyck_counter_task_a_{group['name']}"
    cfg['experiment']['seeds'] = list(group['seeds'])
    cfg['task'].update(
        dyck_pairs=group['dyck_pairs'],
        total_length=group['total_length'],
        seq_len=group['seq_len'],
        repeat_prob=group['repeat_prob'],
        num_noise_tokens=group['num_noise_tokens'],
        prefix_probe_max_len=group['total_length'],
    )
    cfg['training']['batch_size'] = group['batch_size']
    return cfg


def write_group_configs(groups: list[dict]) -> list[Path]:
    GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    paths = []
    for group in groups:
        path = GENERATED_CONFIG_DIR / f"dyck_counter_task_a_{group['name']}.yaml"
        path.write_text(yaml.safe_dump(make_group_config(group), sort_keys=False), encoding='utf-8')
        paths.append(path)
    return paths


config_paths = write_group_configs(TASK_A_GROUPS) if WRITE_CONFIGS else [CONFIG_TEMPLATE]
config_paths

In [ ]:
def pipeline_cmd(config_path: Path, *, seed: int | None = None) -> list[str]:
    cmd = [
        sys.executable,
        str(ROOT / 'scripts' / 'run_pipeline.py'),
        '--config',
        str(config_path),
        '--model',
        'transformer',
        '--num-examples',
        str(NUM_EXAMPLES),
        '--extract-batch-size',
        str(EXTRACT_BATCH_SIZE),
        '--probe-max-rows',
        str(PROBE_MAX_ROWS),
        '--probe-max-classes',
        str(PROBE_MAX_CLASSES),
    ]
    if seed is not None:
        cmd += ['--seed', str(seed)]
    if DEVICE:
        cmd += ['--device', DEVICE]
    return cmd


commands = [pipeline_cmd(path) for path in config_paths]
commands[:3]

In [ ]:
if RUN_TRAINING:
    for cmd in commands:
        print('$', ' '.join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)

## Metric Collection

这里收集三类输出：

1. training/eval behavior：整体 next-token accuracy 和只在 Dyck target 上计算的 accuracy。
2. hidden-state probes：每层 linear readout 预测 left、right、height 和 legal-next class。
3. follow-up probes：oracle forced/free split、output-head alignment、direct intervention、cross-condition transfer、noise-schedule readout 和分桶 diagnostics。


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def collect_task_a_results(results_root: Path = RESULTS_ROOT) -> pd.DataFrame:
    rows = []
    for run_config in sorted(results_root.glob('dyck_counter_task_a_*/*_seed*/config.json')):
        run_dir = run_config.parent
        cfg = load_json(run_config)
        metrics_path = run_dir / 'metrics.json'
        probes_path = run_dir / 'probes' / 'layerwise_probe.csv'
        if not metrics_path.exists() or not probes_path.exists():
            continue
        metrics = load_json(metrics_path)['eval']
        probes = pd.read_csv(probes_path)
        best = probes.sort_values('height_r2', ascending=False).iloc[0].to_dict()
        task = cfg['task']
        rows.append({
            'setting': cfg['setting_name'],
            'model': cfg['model_name'],
            'seed': cfg['seed'],
            'seq_len': task['seq_len'],
            'total_length': task['total_length'],
            'repeat_prob': task['repeat_prob'],
            'num_noise_tokens': task['num_noise_tokens'],
            'loss': metrics['loss'],
            'accuracy': metrics['accuracy'],
            'dyck_accuracy': metrics['dyck_accuracy'],
            'best_layer': int(best['layer']),
            'height_r2': best.get('height_r2'),
            'height_mae': best.get('height_mae'),
            'left_r2': best.get('left_r2'),
            'right_r2': best.get('right_r2'),
            'gap_height_r2_minus_dyck_acc': best.get('height_r2') - metrics['dyck_accuracy'],
        })
    return pd.DataFrame(rows)


summary = collect_task_a_results()
summary

In [ ]:
if not summary.empty:
    display(summary.groupby(['seq_len', 'repeat_prob', 'num_noise_tokens'])[['dyck_accuracy', 'height_r2', 'gap_height_r2_minus_dyck_acc']].mean().reset_index())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    summary.plot.scatter(x='dyck_accuracy', y='height_r2', c='seq_len', colormap='viridis', ax=axes[0])
    axes[0].set_title('Behavior vs hidden count probe')
    axes[0].set_xlabel('Dyck next-token accuracy')
    axes[0].set_ylabel('Best-layer height R2')

    summary.sort_values('seq_len').plot.bar(x='setting', y='gap_height_r2_minus_dyck_acc', ax=axes[1])
    axes[1].set_title('Probe-behavior gap')
    axes[1].set_ylabel('height R2 - dyck accuracy')
    axes[1].tick_params(axis='x', labelrotation=45)
    plt.tight_layout()

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## What We Ran

这次不是比较不同 architecture，而是固定 Transformer 后做 length/noise sweep。所有 run 都完成了训练、hidden extraction、layerwise probe 和 extra probes。

| setting           | seq  | dyck_len | repeat_prob | noise_vocab | steps | batch_size | hidden_examples |
| ----------------- | ---- | -------- | ----------- | ----------- | ----- | ---------- | --------------- |
| tiny_extreme_long | 2000 | 20       | 0.010       | 64          | 15000 | 4          | 512             |
| clean_short       | 48   | 48       | 1.000       | 4           | 15000 | 128        | 4096            |
| noisy_short       | 120  | 48       | 0.500       | 16          | 15000 | 128        | 4096            |
| sparse_medium     | 400  | 200      | 0.250       | 16          | 15000 | 128        | 4096            |
| sparse_long       | 1000 | 400      | 0.250       | 16          | 15000 | 16         | 512             |
| extreme_long      | 2000 | 400      | 0.100       | 64          | 15000 | 4          | 512             |

长序列设置为了显存把 batch size 降低了；`tiny_extreme_long` 和 `extreme_long` 都是 2000 长上下文，但前者只有 20 个括号 token，用来分离“上下文很长”和“counter 本身很长”这两种压力。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Accuracy Definitions

Main Result 表里有四个 behavior accuracy。它们都只关心 next-token 任务里“下一个 token 是 Dyck 括号”的位置；noise target 不进入这些 Dyck accuracy 的分母。

记 `pred_t = argmax(logits_t)`，`target_t = x_{t+1}`，`D = {t: target_t 是 open/close bracket}`，`correct_t = 1[pred_t = target_t]`。实现上 `D` 对应 label 表里的 `target_is_dyck_position=True`。

| metric | 分母 | 计算方法 | 含义 |
|---|---:|---|---|
| `Dyck acc` | training/eval batches 里的 `D` | `mean(correct_t)` | 模型在 Dyck target 上的原始 next-token accuracy。Main Result 表中这个值来自训练阶段保存的 eval metric；与 extra-probe 的 `all_dyck_targets model_acc` 定义相同，但采样 batch 可能不同。 |
| `oracle acc` | extra-probe sampled `D` | `mean(oracle_next_dyck_acc_t)` | 当前 stochastic Dyck generator 下的可预测上限。forced 位置 oracle=1，free 位置 oracle=0.5，所以它取决于 forced/free 的比例。比如约 512 条 eval sequences × 每条 20 个 bracket token ≈ 10240 个 Dyck target positions 做加权平均。 |
| `forced acc` | `D` 中 `forced_state != free` 的位置 | `mean(correct_t | forced)` | 当前 prefix 已经唯一决定下一个括号时，模型是否执行合法 Dyck 约束。`height<=0` 时必须 open；`remaining_opens<=0` 或 `remaining_dyck<=height` 时必须 close。 |
| `free acc` | `D` 中 `forced_state == free` 的位置 | `mean(correct_t | free)` | open 和 close 都合法、sampler 随机二选一时的模型 accuracy。这里 oracle baseline 是 0.5；接近 0.5 不表示模型没学会合法性，而是目标本身没有确定答案。 |

因此读 Main Result 时，核心不是单看 `Dyck acc` 高低，而是比较 `Dyck acc` 是否接近 `oracle acc`，以及失败主要落在 `forced acc` 还是 `free acc`。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Main Result: Two Regimes After Adding `tiny_extreme_long`

原先五个 settings 呈现出一个比较干净的模式：Dyck next-token accuracy 看起来只有大约 0.53-0.62，但 oracle forced/free split 显示它们几乎贴着当前 stochastic generator 的 oracle ceiling。也就是说，这五组的低 accuracy 主要来自 free step 的 50/50 随机性，而不是合法性约束失败。

新加的 `tiny_extreme_long` 则是一个真正不同的 regime：它只有 20 个括号 token 混在 2000 长上下文里，hidden 里仍有可读 count signal（height R2 约 0.80，legal-next probe 约 0.97），但行为没有贴近 oracle。特别是 forced accuracy 只有约 0.26，free accuracy 近乎 0。这说明“上下文极长 + 有效监督极稀疏”会造成比 Markov free-step ceiling 更严重的输出失败。

| setting           | Dyck acc | oracle acc | forced acc | free acc | best height R2 | legal-next probe |
| ----------------- | -------- | ---------- | ---------- | -------- | -------------- | ---------------- |
| tiny_extreme_long | 0.113    | 0.680      | 0.263      | 0.005    | 0.804          | 0.970            |
| clean_short       | 0.603    | 0.597      | 1.000      | 0.501    | 0.966          | 1.000            |
| noisy_short       | 0.622    | 0.608      | 1.000      | 0.501    | 0.894          | 1.000            |
| sparse_medium     | 0.559    | 0.554      | 1.000      | 0.501    | 0.780          | 0.994            |
| sparse_long       | 0.528    | 0.539      | 0.998      | 0.501    | 0.848          | 0.982            |
| extreme_long      | 0.555    | 0.539      | 0.986      | 0.501    | 0.730          | 0.958            |

这张表的核心读法：`forced acc` 代表规则已决定下一步时模型能否执行 Dyck 约束；`free acc` 代表规则没有决定时模型能否猜中随机采样结果。除 `tiny_extreme_long` 外，free acc 稳定在 0.501 左右，正好说明这一部分没有可学习的确定目标；`tiny_extreme_long` 的 forced/free 都低，说明它不是 oracle ceiling 问题，而是稀疏信号下的行为读出失败。

<!-- TASK_A_SPARSE_SUPERVISION_ABLATION -->
## Sparse Supervision Ablation: Fixed `seq_len=2000`

这里不改 Dyck 生成规则，只固定 `seq_len=2000` 和 `noise_vocab=64`，扫已完成的 bracket token 数量：`[20, 24, 28, 32, 34, 36, 40, 44, 48, 56, 64, 80, 100, 200, 400]`。目的就是看 `tiny_extreme_long` 的失败到底来自 2000 长上下文本身，还是来自 Dyck target 在训练 loss 里太稀疏。

| brackets | density | Dyck acc | oracle | forced | free  | height R2 | legal-next |
| -------- | ------- | -------- | ------ | ------ | ----- | --------- | ---------- |
| 20       | 0.010   | 0.098    | 0.680  | 0.263  | 0.005 | 0.804     | 0.970      |
| 24       | 0.012   | 0.063    | 0.663  | 0.187  | 0.004 | 0.822     | 0.965      |
| 28       | 0.014   | 0.071    | 0.647  | 0.221  | 0.009 | 0.815     | 0.975      |
| 32       | 0.016   | 0.071    | 0.640  | 0.236  | 0.007 | 0.793     | 0.867      |
| 34       | 0.017   | 0.125    | 0.634  | 0.447  | 0.007 | 0.777     | 0.971      |
| 36       | 0.018   | 0.169    | 0.633  | 0.602  | 0.013 | 0.783     | 0.966      |
| 40       | 0.020   | 0.226    | 0.625  | 0.777  | 0.043 | 0.755     | 0.859      |
| 44       | 0.022   | 0.215    | 0.621  | 0.760  | 0.042 | 0.753     | 0.871      |
| 48       | 0.024   | 0.227    | 0.612  | 0.826  | 0.054 | 0.779     | 0.894      |
| 56       | 0.028   | 0.305    | 0.606  | 0.927  | 0.137 | 0.764     | 0.903      |
| 64       | 0.032   | 0.566    | 0.600  | 0.977  | 0.463 | 0.732     | 0.897      |
| 80       | 0.040   | 0.577    | 0.590  | 0.962  | 0.492 | 0.781     | 0.902      |
| 100      | 0.050   | 0.578    | 0.578  | 0.988  | 0.503 | 0.685     | 0.897      |
| 200      | 0.100   | 0.554    | 0.556  | 0.990  | 0.499 | 0.733     | 0.938      |
| 400      | 0.200   | 0.539    | 0.539  | 0.986  | 0.501 | 0.730     | 0.958      |

结果现在更具体：20、24、28、32 个括号时，forced accuracy 仍然只有约 0.19-0.26，说明模型没有稳定执行强制 open/close 规则；34 个括号时 forced accuracy 抬到约 0.45，是转折的前沿；36 个括号时跳到约 0.60，40-48 个括号已经到约 0.76-0.83。56 个括号时 forced accuracy 已经到约 0.93，说明 forced-rule readout 基本恢复。但 free accuracy 要到 64 个括号才从 0.14 跳到约 0.46，并在 80/100/200/400 附近稳定到约 0.49-0.50。因此当前 seed=0 下有两段变化：forced 规则执行的突变大致在 34-40 brackets，free/random target 行为接近 oracle 的突变大致在 56-64 brackets。到 80-100 个括号时，Dyck accuracy 已经接近 200/400 个括号的长上下文结果。与此同时 hidden probe 一直不低，尤其 legal-next probe 在所有密度下都很高。所以更合理的解释是：`tiny_extreme_long` 的主要瓶颈不是 2000 长度本身，而是 Dyck supervision 在 next-token loss 中过于稀疏，导致输出头没有稳定学会 bracket readout。

<!-- TASK_A_SPARSE_SUPERVISION_ABLATION -->
![sparse_supervision_ablation](../../figures/dyck_counter_sparse_supervision_ablation/sparse_supervision_ablation.png)

<!-- TASK_A_REWRITTEN_ANALYSIS -->
### Behavior / Probe Overview

![task_a_overview](../../figures/dyck_counter_task_a/task_a_overview.png)

![task_a_height_axis_projection](../../figures/dyck_counter_task_a/task_a_height_axis_projection.png)

这些图说明 hidden 里确实有可读的 count/height structure：clean/noisy short 最强，长序列和更稀疏噪声下 R2 下降但仍明显高于行为 accuracy。height-axis projection 也显示表征沿 count 方向有连续结构，而不是 probe 偶然捡到噪声。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Probe 1: Oracle Forced/Free Split

这个 probe 把 Dyck target 分成两类：

- forced：由当前 prefix 和 balanced-parentheses 约束唯一决定下一步，oracle accuracy = 1。
- free：open 和 close 都合法，当前 sampler 随机采样，oracle accuracy = 0.5。

| setting           | model forced | model free | forced_oracle | free_oracle |
| ----------------- | ------------ | ---------- | ------------- | ----------- |
| tiny_extreme_long | 0.263        | 0.005      | 1.000         | 0.500       |
| clean_short       | 1.000        | 0.501      | 1.000         | 0.500       |
| noisy_short       | 1.000        | 0.501      | 1.000         | 0.500       |
| sparse_medium     | 1.000        | 0.501      | 1.000         | 0.500       |
| sparse_long       | 0.998        | 0.501      | 1.000         | 0.500       |
| extreme_long      | 0.986        | 0.501      | 1.000         | 0.500       |

结果分成两类：原先五组 forced 位置几乎全对、free 位置稳定在随机上限；`tiny_extreme_long` 则 forced 也明显失败。因此，原始 Dyck accuracy 不是一个纯粹的 algorithmic counting 指标；它既会混合“可由规则决定的合法性”和“生成器随机 coin flip”，也会在极稀疏长上下文中暴露真正的 behavior/readout failure。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![extra_probe_oracle_forced_free](../../figures/dyck_counter_task_a_extra_probes/extra_probe_oracle_forced_free.png)

右图还能解释为什么原先五组的整体 accuracy 会随着设置变化：不同长度/稀疏度下 forced/free 的比例不同。free rows 占大多数时，总体 accuracy 会自然靠近 0.5。`tiny_extreme_long` 额外说明了一点：当 Dyck token 在 2000 长上下文里过于稀疏时，模型甚至没有稳定学会 forced rows。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Probes 2-3: Is The Counter Direction Directly Used By The Output Head?

这里问的是更强的因果问题：height 在线性 probe 中可读，并不自动说明 output head 正沿着同一个方向做 close-vs-open 决策。所以我们做了两步：先比较 final-layer height direction 和 output head 的 close-minus-open 向量，再沿 height direction 直接移动 final hidden，看 bracket probability 是否系统变化。

| setting | final_cos | axis-margin_corr | P(close/bracket)_at_delta_0 | margin_shift/axis_std |
| --- | --- | --- | --- | --- |
| tiny_extreme_long | -0.021 | 0.330 | 0.504 | -0.006 |
| clean_short | -0.065 | 0.344 | 0.496 | -0.003 |
| noisy_short | -0.015 | 0.376 | 0.496 | -0.002 |
| sparse_medium | -0.042 | 0.223 | 0.498 | -0.008 |
| sparse_long | -0.017 | 0.183 | 0.491 | -0.003 |
| extreme_long | -0.006 | 0.105 | 0.501 | -0.001 |

结果偏谨慎：cosine 基本接近 0 且略负；axis-margin correlation 为正，说明沿数据流形 height 和 close/open margin 有相关性；但 direct final-hidden intervention 几乎不改变 `P(close | bracket logits)`。这支持一个更细的说法：counter 在 hidden state 中可读，但 output head 并没有简单地把 probe 找到的 height 方向当作直接控制旋钮。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![extra_probe_output_head_causal](../../figures/dyck_counter_task_a_extra_probes/extra_probe_output_head_causal.png)

这个 intervention 只是 final hidden 上的 direct-logit intervention，不等价于在中间层改激活后重新 forward。下一节补的是更接近 forward computation 的 layer-wise activation patch。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Ablation: Does Removing The Height Direction Matter?

为了检验 probe direction 是否是 output head 真正在用的方向，我们在 final hidden 上做 ablation：把每个 hidden state 沿 final-layer height probe direction 的投影移除，再重新过同一个 output head。同时脚本里还保存了 shuffle-height-axis 和 remove-random-direction 两个对照。

| setting           | delta acc all | delta acc forced | delta acc free | delta NLL all |
| ----------------- | ------------- | ---------------- | -------------- | ------------- |
| tiny_extreme_long | 0.0007        | 0.0019           | 0.0000         | -0.0009       |
| clean_short       | 0.0005        | 0.0000           | 0.0006         | -0.0000       |
| noisy_short       | 0.0001        | 0.0000           | 0.0001         | -0.0001       |
| sparse_medium     | -0.0002       | 0.0000           | -0.0003        | -0.0001       |
| sparse_long       | 0.0005        | 0.0000           | 0.0005         | -0.0000       |
| extreme_long      | -0.0002       | 0.0002           | -0.0002        | 0.0000        |

结果基本是负结果：移除 height direction 后，Dyck-target accuracy 和 NLL 几乎不变。这和前面的 output-head cosine/direct intervention 一致，说明当前 linear height probe 找到的方向主要是可读表征方向，而不是 final logits 的直接控制方向。所以这里不能把“height R2 高”直接解释成“输出头沿这个方向执行计数决策”。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![height_direction_ablation](../../figures/dyck_counter_task_a_ablation/height_direction_ablation.png)

<!-- TASK_A_ACTIVATION_PATCH -->
## Layer-Wise Steering: Does The Counter Direction Control The Forward Pass?

前面的 direct intervention 只是在 final hidden 上改 logits 前的向量。这里补了更接近因果机制的 patch：在某一层的 sequence activation 上沿该层 height probe direction 加/减若干 axis std，然后继续跑后续 Transformer layers 和 output head。当前先对固定 `seq_len=2000` 的 sparse ladder 做小样本 smoke test；为了避免同一序列里多个 patch 互相污染，每条 sampled sequence 只选一个 Dyck target prefix。

| setting             | brackets | best layer | height slope | random slope |
| ------------------- | -------- | ---------- | ------------ | ------------ |
| tiny_extreme_long   | 20       | 0          | 0.0013       | -0.0010      |
| sparse_len2000_b48  | 48       | 2          | -0.0011      | 0.0030       |
| sparse_len2000_b200 | 200      | 0          | 0.0010       | -0.0010      |
| extreme_long        | 400      | 1          | 0.0008       | 0.0001       |

结果仍然是负向的：`P(close | bracket logits)` 对 height-direction patch 的斜率只有约 0.001 量级，和 random-direction control 同量级。这个小样本结果不能替代正式 multi-seed 大样本统计，但它没有支持“probe 找到的 counter direction 可以在 forward computation 中直接控制 open/close 决策”。更像是 counter 信息可读，但模型实际决策可能使用了分布式、非线性或不同坐标的特征。

<!-- TASK_A_ACTIVATION_PATCH -->
![layerwise_activation_patch](../../figures/dyck_counter_task_a_activation_patch/layerwise_activation_patch.png)

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Probes 4-6: Transfer, Noise Schedule, And Binned Diagnostics

这组 probe 回答三个问题：不同设置学到的 counter 坐标系是否共享；hidden state 是否线性编码“下一个 token 是不是 Dyck”；错误集中在哪些 height/position 区域。

| setting           | self-transfer R2 | next-is-dyck over baseline | next-symbol over baseline |
| ----------------- | ---------------- | -------------------------- | ------------------------- |
| tiny_extreme_long | 0.851            | -0.000                     | 0.000                     |
| clean_short       | 0.980            | N/A                        | 0.097                     |
| noisy_short       | 0.956            | 0.004                      | 0.003                     |
| sparse_medium     | 0.907            | 0.018                      | -0.004                    |
| sparse_long       | 0.908            | -0.006                     | -0.005                    |
| extreme_long      | 0.827            | -0.003                     | -0.003                    |

cross-condition transfer 的结论是：每个 run 自己的 height probe 都能工作，但跨设置迁移大多很差，只有 `sparse_medium` 和 `sparse_long` 之间有一点正迁移。这说明 counter geometry 目前更像是每个训练条件内部的局部坐标，而不是已经对齐成一个通用坐标系。

noise-schedule probe 加入 majority baseline 后，next-is-dyck / next-symbol 的优势基本接近 0。`clean_short` 的 next-symbol 有一个小例外，因为它没有 noise 插入，target type 退化成 bracket symbol readout；其余 noisy/sparse 设置没有显示出稳定的线性 schedule signal。

binned diagnostics 则把行为现象定位得更具体：原先五组里，height=0 的 forced-open 区域几乎全对，height>0 后大部分 free 区域回到约 0.5；`tiny_extreme_long` 的分桶则显示稀疏长上下文下 forced 区域也会失败。序列后段 accuracy 上升时，通常来自必须 close 的 forced 区域变多。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![extra_probe_transfer_noise_bins](../../figures/dyck_counter_task_a_extra_probes/extra_probe_transfer_noise_bins.png)

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Overall Interpretation

这次 Task A 的结论可以压缩成三句话。

第一，Transformer 确实学到了 Dyck prefix counter 的 hidden-state representation。`left/right/height` 都能被线性 probe 读出，legal-next class 也几乎可读。

第二，当前原始 Dyck next-token accuracy 不是衡量 counter 能力的好单一指标。原先五组里，Markov-style sampler 的 free step 本来就是随机的，模型整体 accuracy 几乎等于 oracle forced/free baseline；但 `tiny_extreme_long` 显示，在 2000 长上下文里只放 20 个括号时，模型连 forced 位置也不能稳定输出。稀疏监督 ablation 进一步说明：20-32 brackets 仍失败，34 brackets 是 forced 恢复的前沿，36-48 brackets 进入快速恢复区间，56 brackets 后 forced 规则基本能执行；但 free/random target 直到 64 brackets 才接近 oracle baseline。所以主要瓶颈是 Dyck target 在训练中太稀疏，当前 seed=0 下更像有两段转折：forced readout 在 34-40，整体 Dyck/free 行为在 56-64。

第三，虽然 counter 可读，但它在不同 length/noise 条件下没有形成统一坐标系；final output head 没有简单沿 probe height direction 使用它，移除该方向的 ablation 也几乎不改变行为。layer-wise height-direction patch smoke test 也没有看到 height direction 对 `P(close)` 的系统控制。cross-model patch 则补充了另一面：好模型的最后层 full activation patch 到坏模型后，forced 行为可以明显改善；donor-bank retrieval 也能改善坏 recipient。但只替换 height scalar 基本无效，而且 state-random retrieval 经常和 matched retrieval 一样强。因此当前结果支持“行为相关信息存在于更完整的 hidden state / activation distribution 中”，但还不能支持“线性 height probe scalar 就是可直接因果控制的通用计数旋钮”。

## CountScope-style Online Activation Patching

<!-- TASK_A_COUNTSCOPE_ONLINE_PATCHING -->

这一节参考 reading notes 里 activation patching / CountScope 的做法，重新补一个更谨慎的 Task A patching。核心变化是：不再把不同模型的 activation 直接互换作为主证据，而是在同一个模型内构造 source/target/patched 三个上下文，在 target forward pass 的某一层在线替换 activation，然后让后续 Transformer layer 继续计算。

这次只跑固定 2000 context、noise vocab 64 的三个 sparse setting：`b20`、`b48`、`b100`。每个 setting 收集 `local_interchange` 和 `future_continued` 各 256 个 patch pair。

### 实验设计

统一符号如下：

- `C` 是 source sequence，提供被 patch 的 activation。
- `C'` 是 target sequence，提供正常 forward 的上下文和原本支持的 target bracket。
- `C*` 是 patched target：在 target forward pass 的某一层，把指定 position 的 hidden state 替换成 source activation，然后继续跑后续 layers 和 output head。

这里的 `position` 都指当前 hidden state 所在的位置；模型用这个位置的 hidden state 预测下一 token。`target token` 是原 target sequence 真实下一 bracket；`hypothesis token` 是 patch 后我们希望模型转向的 bracket。

#### 1. `local_interchange`: same-position forced decision patch

**构造 pair。** 在 source 和 target 中各选一个 forced Dyck next-token position。`forced` 表示当前 Dyck 状态已经唯一决定下一 bracket：要么必须 open，要么必须 close。我们只保留 source forced bracket 与 target forced bracket 相反的 pair，例如 target 必须 open、source 必须 close。

**怎么 patch。** 先正常 forward target 得到 `C'`。然后在 target 的同一个 prediction position 上，把第 `l` 层 hidden state 替换成 source 在对应 forced position 的第 `l` 层 hidden state，得到 `C*`，再继续跑后续 Transformer layers 和 output head。

**怎么评估。** eval position 就是被 patch 的这个 position。如果 target 原本应该预测 open、source 表示 close，那么 hypothesis token 就是 close。我们看 patch 后 close/open readout 是否从 target token 转向 hypothesis token。

**这个实验回答什么。** 它回答的是：某层 hidden state 进入模型自己的后续 computation 和 output head 后，是否足以局部改变当前 forced open/close decision。所以这是一个局部 causal decoding / interchange test。

**它不能单独说明什么。** 如果这个实验阳性，只能说明 activation 中存在会被 output path 使用的 bracket-decision 信息；它还不能证明 source 的完整 counter state 被 target 后续过程继续使用。

#### 2. `future_continued`: earlier-state patch plus later forced decision

**构造 pair。** 在 target 中先选一个较早的 Dyck bracket position 作为 patch position，再选同一 target sequence 后面一个 forced Dyck next-token position 作为 eval position。source 提供 patch position 的 activation。

**怎么定义 continued hypothesis。** 这里 hypothesis 不是 source 当前下一 token。我们把 source 在 patch position 的计数状态当作新的起点，然后接上 target 从 patch position 到 eval position 之间实际发生的 Dyck 增量：

`continued_left = source_left + (target_eval_left - target_patch_left)`

`continued_right = source_right + (target_eval_right - target_patch_right)`

`continued_dyck_seen = source_dyck_seen + (target_eval_dyck_seen - target_patch_dyck_seen)`

然后用这个 continued state 判断 eval position 是否 forced open 或 forced close。如果 continued state 给出的 forced token 与 target 原本 token 不同，这个 pair 才保留。

**怎么 patch。** 在 target 的较早 patch position 替换第 `l` 层 hidden state，然后让模型继续计算后续 layers。最后不是在 patch position 解码，而是在后面的 eval position 解码。

**怎么评估。** 如果 patched counter state 真的被 target 后续 computation 当作新的计数起点继续使用，那么 eval position 的输出应该从 target token 向 continued hypothesis token 移动。

**这个实验回答什么。** 它更接近 notes 里的 continued counting：patch 的不是当前 readout，而是较早 latent state；真正关心的是这个 latent state 是否会影响后面位置的 forced decision。

**它比 `local_interchange` 更严格。** `local_interchange` 只需要 patch 改变当前 output readout；`future_continued` 要求 patch 进去的状态被后续 sequence computation 保持并使用。所以如果 `local_interchange` 阳性但 `future_continued` 弱，比较合理的解释是：模型 late hidden 能局部控制 bracket readout，但还没有强证据说明它在做可迁移的 continued counter-state computation。

### 指标定义

先固定三个对象：

- `C'`：target sequence 的正常 forward，不做 patch。
- `C*`：patched target，也就是把 source activation 放进 target 后继续 forward。
- `target token`：target sequence 原本真实的下一 bracket。
- `hypothesis token` / `hyp`：如果 source activation 起作用，我们希望模型转向的 bracket。在 `local_interchange` 里它就是 source forced bracket；在 `future_continued` 里它是 continued state 算出的 forced bracket。

还要区分两种概率空间：

- `P_bracket(x)`：只拿 `close` 和 `open` 两个 logits 做 softmax 后得到的概率。它只问：如果已经限定下一 token 必须是 bracket，模型更偏 close 还是 open？
- `P_full(x)`：在完整 vocab 上做 softmax 后得到的概率。它问：模型最终 next-token 输出真的会不会选这个 bracket token；这里 noise token 也参与竞争。

#### `bracket-normalized CI`

CI 是 causal influence score，衡量 patch 是否把概率从 target token 推向 hypothesis token。bracket-normalized 版本用 `P_bracket`：

```text
CI_bracket = 0.5 * [
    P_bracket(hyp, C*)    - P_bracket(hyp, C')
  + P_bracket(target, C') - P_bracket(target, C*)
]
```

读法：

- `CI_bracket > 0`：patch 后 close/open 子空间向 hypothesis 移动。
- `CI_bracket ≈ 0`：patch 基本没有改变 close/open 决策。
- `CI_bracket < 0`：patch 反而让模型更偏 target 或相反方向。
- 这个指标**忽略 noise token**，所以它只能说明 open/close 相对排序是否被改变。

#### `full-vocab CI`

公式一样，但把 `P_bracket` 换成 `P_full`：

```text
CI_full = 0.5 * [
    P_full(hyp, C*)    - P_full(hyp, C')
  + P_full(target, C') - P_full(target, C*)
]
```

读法：

- `CI_full > 0`：patch 在完整 next-token 分布里也把概率推向 hypothesis。
- 如果 `CI_bracket` 很高但 `CI_full` 很低，说明 patch 改变了 close/open 的相对排序，但 bracket token 本身仍然没有在 full vocab 里赢过 noise token。这正是 b20 这类 sparse setting 需要单独检查的问题。

#### `ci_minus_self`

`target_self` 是无操作 control：把 target 自己同一位置的 activation 再 patch 回 target。理论上这不该改变输出，但实际代码路径里仍可能有极小数值误差。

```text
ci_minus_self = CI(mode) - CI(target_self)
```

所以后面主要看 `ci_minus_self`，而不是裸 CI。它表示扣掉 patching wrapper 自身误差后的净效应。

#### `patched_hypothesis_acc`

patch 后，只在 close/open 两个 bracket logits 里取 argmax：

```text
patched_hypothesis_acc = mean(argmax_close_open(C*) == hypothesis token)
```

读法：它回答“patch 后，如果只在 open/close 中二选一，模型是否选了 hypothesis”。它高说明 bracket 子空间被 patch 成功翻转，但不保证 full vocab 最终输出 hypothesis。

#### `patched_full_hypothesis_acc`

patch 后，在完整 vocab 里取 argmax：

```text
patched_full_hypothesis_acc = mean(argmax_full_vocab(C*) == hypothesis token)
```

读法：它比 `patched_hypothesis_acc` 更严格。只有当 hypothesis bracket token 真的赢过所有 noise token 和另一个 bracket token 时，才算正确。

#### 结果表里另外几个字段

- `patched_target_acc`：patch 后，close/open 子空间 argmax 仍等于 target token 的比例。如果 patch 有效，这个值通常应该下降。
- `patched_full_target_acc`：patch 后，完整 vocab argmax 仍等于 target token 的比例。
- `mean_delta_p_hypothesis`：patch 前后 hypothesis 的 `P_bracket` 平均变化，即 `P_bracket(hyp, C*) - P_bracket(hyp, C')`。越大表示 patch 越提高 hypothesis bracket 概率。
- `mean_delta_full_p_hypothesis`：同上，但用完整 vocab 概率 `P_full`。它能看出 patch 是否真的提高了 hypothesis token 在完整 next-token 分布里的概率。

简化读法：

- 想看 open/close 机制有没有被局部控制：看 `ci_minus_self` 和 `patched_hypothesis_acc`。
- 想看最终 next-token 输出有没有真的变：看 `full_vocab_ci_minus_self` 和 `patched_full_hypothesis_acc`。
- 想区分“open/close 方向对了”还是“bracket token 输给 noise”：比较 bracket 指标和 full-vocab 指标。

### Main Patch Results: Source-state Mode

| experiment | setting | layer | n | ci_minus_self | full_vocab_ci_minus_self | patched_hypothesis_acc | patched_full_hypothesis_acc | patched_target_acc | patched_full_target_acc | mean_delta_p_hypothesis | mean_delta_full_p_hypothesis |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| future_continued | b100 | 0 | 256 | 0.01 | -0.0001 | 0.0234 | 0.0195 | 0.9766 | 0.957 | 0.01 | 0.0004 |
| future_continued | b100 | 1 | 256 | 0.0002 | -0 | 0.0078 | 0.0078 | 0.9922 | 0.9688 | 0.0002 | 0.0002 |
| future_continued | b100 | 2 | 256 | 0 | 0 | 0 | 0 | 1 | 0.9922 | 0 | 0 |
| future_continued | b20 | 0 | 256 | 0.1379 | 0.001 | 0.1523 | 0 | 0.8477 | 0.3711 | 0.1379 | 0.0007 |
| future_continued | b20 | 1 | 256 | 0.0379 | 0.0012 | 0.0547 | 0 | 0.9453 | 0.3516 | 0.0379 | 0.0003 |
| future_continued | b20 | 2 | 256 | 0 | 0 | 0.0156 | 0 | 0.9844 | 0.4219 | 0 | 0 |
| future_continued | b48 | 0 | 256 | 0.0411 | -0.0001 | 0.0547 | 0 | 0.9453 | 0.7188 | 0.0411 | 0.0004 |
| future_continued | b48 | 1 | 256 | 0.0112 | 0.0002 | 0.043 | 0.0078 | 0.957 | 0.7656 | 0.0112 | 0.0004 |
| future_continued | b48 | 2 | 256 | 0 | 0 | 0.0117 | 0.0039 | 0.9883 | 0.8203 | 0 | 0 |
| local_interchange | b100 | 0 | 256 | 0.0418 | 0.0023 | 0.0234 | 0.0234 | 0.9766 | 0.9688 | 0.0418 | 0.0027 |
| local_interchange | b100 | 1 | 256 | 0.328 | 0.0187 | 0.4336 | 0.3789 | 0.5664 | 0.4609 | 0.328 | 0.0145 |
| local_interchange | b100 | 2 | 256 | 0.7442 | 0.0374 | 0.9883 | 0.9844 | 0.0117 | 0.0078 | 0.7442 | 0.0378 |
| local_interchange | b20 | 0 | 256 | 0.0059 | 0 | 0.0234 | 0 | 0.9766 | 0.2656 | 0.0059 | 0.0002 |
| local_interchange | b20 | 1 | 256 | 0.1674 | 0.0038 | 0.1328 | 0.0234 | 0.8672 | 0.082 | 0.1674 | 0.0028 |
| local_interchange | b20 | 2 | 256 | 0.8801 | 0.0154 | 0.9961 | 0.25 | 0.0039 | 0 | 0.8801 | 0.0164 |
| local_interchange | b48 | 0 | 256 | 0.0129 | 0.0008 | 0.0156 | 0.0078 | 0.9844 | 0.7773 | 0.0129 | 0.0003 |
| local_interchange | b48 | 1 | 256 | 0.2909 | 0.0107 | 0.3008 | 0.0625 | 0.6992 | 0.2734 | 0.2909 | 0.0067 |
| local_interchange | b48 | 2 | 256 | 0.7818 | 0.0261 | 0.9805 | 0.875 | 0.0195 | 0.0078 | 0.7818 | 0.026 |

![local_interchange_ci](../../figures/dyck_counter_task_a_countscope_patching/local_interchange_ci.png)

![future_continued_ci](../../figures/dyck_counter_task_a_countscope_patching/future_continued_ci.png)

### Controls

controls 的读法很重要：

- `source_state` 是主 patch：source activation 的 forced token 与 target 相反。
- `source_state_shuffle` 保留 source activation distribution，但打乱 source-target 对应关系。
- `mean_forced` 用 forced positions 的均值 activation 替换，是 in-distribution-ish mean control。
- `zero` 是 OOD control，只用于判断 patch sensitivity，不能当机制证据。

如果 `source_state` 明显强于 shuffle/mean/zero，才支持逐样本 source state 被语义性移植。如果 source_state 和 shuffle 接近，则更像 activation distribution 或 late readout 被扰动，而不是 matched counter state transfer。

| experiment | setting | mode | layer | n | ci_minus_self | full_vocab_ci_minus_self | patched_hypothesis_acc | patched_full_hypothesis_acc |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| future_continued | b100 | mean_forced | 0 | 256 | 0.014 | 0.0012 | 0.0078 | 0.0078 |
| future_continued | b100 | source_state | 0 | 256 | 0.01 | -0.0001 | 0.0234 | 0.0195 |
| future_continued | b100 | source_state_shuffle | 0 | 192 | 0.0104 | -0.0003 | 0.0208 | 0.0156 |
| future_continued | b100 | zero | 0 | 256 | 0.018 | -0.0002 | 0.0117 | 0.0117 |
| future_continued | b20 | mean_forced | 0 | 256 | 0.1316 | 0.0036 | 0.1328 | 0 |
| future_continued | b20 | source_state | 0 | 256 | 0.1379 | 0.001 | 0.1523 | 0 |
| future_continued | b20 | source_state_shuffle | 0 | 146 | 0.2167 | 0.0016 | 0.2466 | 0 |
| future_continued | b20 | zero | 0 | 256 | 0.1438 | 0.0011 | 0.125 | 0.0117 |
| future_continued | b48 | mean_forced | 0 | 256 | 0.0375 | 0.0012 | 0.0625 | 0.0078 |
| future_continued | b48 | source_state | 0 | 256 | 0.0411 | -0.0001 | 0.0547 | 0 |
| future_continued | b48 | source_state_shuffle | 0 | 166 | 0.0489 | -0 | 0.0422 | 0.006 |
| future_continued | b48 | zero | 0 | 256 | 0.0502 | 0.0006 | 0.043 | 0.0078 |
| local_interchange | b100 | mean_forced | 2 | 256 | 0.4204 | 0.0197 | 0.6055 | 0.3438 |
| local_interchange | b100 | source_state | 2 | 256 | 0.7442 | 0.0374 | 0.9883 | 0.9844 |
| local_interchange | b100 | source_state_shuffle | 2 | 148 | 0.7385 | 0.0365 | 0.9865 | 0.9797 |
| local_interchange | b100 | zero | 1 | 256 | 0.3772 | 0.0191 | 0.5273 | 0.4766 |
| local_interchange | b20 | mean_forced | 2 | 256 | 0.5315 | 0.0081 | 0.6367 | 0.0312 |
| local_interchange | b20 | source_state | 2 | 256 | 0.8801 | 0.0154 | 0.9961 | 0.25 |
| local_interchange | b20 | source_state_shuffle | 2 | 142 | 0.8822 | 0.0156 | 1 | 0.2676 |
| local_interchange | b20 | zero | 1 | 256 | 0.5134 | 0.0098 | 0.6406 | 0.1484 |
| local_interchange | b48 | mean_forced | 2 | 256 | 0.4614 | 0.0143 | 0.6328 | 0.1016 |
| local_interchange | b48 | source_state | 2 | 256 | 0.7818 | 0.0261 | 0.9805 | 0.875 |
| local_interchange | b48 | source_state_shuffle | 2 | 144 | 0.8002 | 0.0273 | 0.9792 | 0.8889 |
| local_interchange | b48 | zero | 1 | 256 | 0.4121 | 0.0138 | 0.5547 | 0.0703 |

![patching_controls_summary](../../figures/dyck_counter_task_a_countscope_patching/patching_controls_summary.png)

![patching_full_vocab_controls_summary](../../figures/dyck_counter_task_a_countscope_patching/patching_full_vocab_controls_summary.png)

### 结果说明

**1. `local_interchange`: layer 越靠后，局部 open/close readout 越容易被 patch。**

这个任务是在同一个 forced position 上 patch，然后马上解码，所以它主要测当前位置 hidden 对 output readout 的因果作用。source-state 的 bracket CI 随 layer 增强：b20 `L0=0.006/L1=0.167/L2=0.880`；b48 `L0=0.013/L1=0.291/L2=0.782`；b100 `L0=0.042/L1=0.328/L2=0.744`。因此主文报 layer 2，是因为 layer 2 最接近 output head、效应最大，最直接回答“late hidden 是否能控制 bracket readout”。这不是说 layer 1 没有效果；layer 1 已经有中等效应，但尚未达到 layer 2 的直接 readout 强度。

layer 2 的 bracket 指标是强阳性：b20=`0.880`, b48=`0.782`, b100=`0.744`；bracket-subspace hypothesis accuracy 也接近 1。结论：最后层 hidden 中存在 output path 能直接读出的 open/close decision 信息。

**2. `local_interchange` 的 full-vocab 结果更保守，尤其 b20。**

同样是 layer 2，full-vocab CI 远小于 bracket CI：b20=`0.015`, b48=`0.026`, b100=`0.037`。b20 的 bracket 子空间几乎被翻转，但 full-vocab hypothesis accuracy 只有 `0.250`；b48/b100 是 `0.875`、`0.984`。解释：b20 的 open/close 排序可以被 patch 改动，但 bracket token 在完整 vocab 里仍常输给 noise token；b48/b100 的 full-vocab readout 明显更稳定。

**3. `future_continued`: 没有稳定的 continued-counting transfer。**

这个任务把 source state patch 到 target 较早位置，再看后面 forced position 是否转向 continued hypothesis。这里 layer 0/1 才有机会通过后续 Transformer layers 影响 later position；layer 2 patch 到较早 position 后已经没有后续 sequence mixing，所以 layer 2 接近 0 是预期的。source-state bracket CI 为：b20 `L0=0.138/L1=0.038/L2=0.000`；b48 `L0=0.041/L1=0.011/L2=0.000`；b100 `L0=0.010/L1=0.000/L2=0.000`。最有机会传播的 layer 0 也只有：b20=`0.138`, b48=`0.041`, b100=`0.010`，full-vocab CI 基本接近 0。这说明 earlier patch 对后续 forced decision 的影响弱，不能支持“source counter state 被继续使用”的强说法。

**4. Controls 限制了机制解释。**

`local_interchange` 中，source_state 和 source_state_shuffle 很接近：b20 source=`0.880` vs shuffle=`0.882`；b48 source=`0.782` vs shuffle=`0.800`；b100 source=`0.744` vs shuffle=`0.738`。因此 local 阳性更像“patch 到了一个 readout-aligned activation distribution”，而不是严格的 matched source-target semantic transfer。`future_continued` 中，shuffle/mean/zero controls 也经常接近或超过 source_state，因此 continued-counting 证据更弱。

**结论。** 当前结果支持：late hidden 对当前位置 close/open readout 有强因果作用。当前结果不支持：source counter state 被 target 后续 computation 稳定继续使用。b20 的额外问题是 full-vocab bracket-vs-noise 竞争，而不是单纯 open/close 子空间失效。

## Extended Patching and Sparse Failure Diagnostics

<!-- TASK_A_EXTENDED_PATCHING_GEOMETRY -->

这一节补两个更精细的问题。

第一，前面的 full-state online patch 能改变 bracket readout，但还不能说明是哪条方向在起作用。所以这里做 axis/span patching：只把 source-target 在某个方向或低维子空间上的投影 copy 到 target，然后继续后续 layers。

第二，固定 `seq_len=2000` 改变 bracket sparsity 时，低 density 模型为什么差？这里同时看 full-vocab CE、bracket-only CE、bracket token mass、方向夹角，以及 final-hidden direction ablation。

### Axis/Span Patching: 设计

仍然使用 same-model online patching：target/source 都选 forced Dyck next-token position，且 source 的 forced token 与 target 相反。patch 后继续后续 Transformer layers，然后在同一 position 解码。

| mode | definition |
| --- | --- |
| full_source | 整段 source hidden state 替换 target hidden state。 |
| full_source_shuffle | 打乱 source-target pairing 的 full-state control。 |
| output_close_open_scalar | 只 copy source 在 output head 的 close-minus-open 方向上的 scalar projection。 |
| output_bracket_noise_scalar | 只 copy source 在 bracket-vs-noise 输出方向上的 scalar projection。 |
| output_output2d_span | copy close-open 和 bracket-vs-noise 两维 output span。 |
| height_scalar | 只 copy ridge height probe 方向上的 projection。 |
| left/right/height-left-right | copy left/right count probe 方向或它们和 height 构成的 span。 |
| current_bracket_scalar | copy 当前 token 是 close vs open 的 hidden mean-difference 方向。 |
| forced_next_scalar | copy forced next close vs forced next open 的 hidden mean-difference 方向。 |

`bracket CI` 只在 close/open 两个 logits 上 softmax；`full-vocab CI` 在完整 vocab softmax 上计算。如果某个方向 bracket CI 高但 full-vocab CI 低，说明它能控制 open/close 相对排序，但未必能让 bracket token 赢过 noise token。

### Axis/Span Patching: 结果

| setting | bracket_tokens | mode | layer | n | ci_minus_self | full_vocab_ci_minus_self | patched_hypothesis_acc | patched_full_hypothesis_acc |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| b20 | 20 | forced_next_scalar | 2 | 256 | 0.8719 | 0.0183 | 0.9961 | 0.375 |
| b20 | 20 | full_source | 2 | 256 | 0.8693 | 0.0152 | 0.9961 | 0.2617 |
| b20 | 20 | full_source_shuffle | 2 | 136 | 0.882 | 0.0167 | 1 | 0.25 |
| b20 | 20 | height_scalar | 1 | 256 | 0.0275 | 0.0008 | 0.0234 | 0 |
| b20 | 20 | output_close_open_scalar | 2 | 256 | 0.8693 | 0.0202 | 0.9961 | 0.3711 |
| b48 | 48 | forced_next_scalar | 2 | 256 | 0.7748 | 0.0311 | 0.9609 | 0.6719 |
| b48 | 48 | full_source | 2 | 256 | 0.7747 | 0.026 | 0.9805 | 0.8008 |
| b48 | 48 | full_source_shuffle | 2 | 128 | 0.7841 | 0.0256 | 0.9766 | 0.8359 |
| b48 | 48 | height_scalar | 1 | 256 | 0.006 | 0.0002 | 0.0234 | 0.0156 |
| b48 | 48 | output_close_open_scalar | 2 | 256 | 0.7747 | 0.0313 | 0.9805 | 0.6562 |
| b100 | 100 | forced_next_scalar | 2 | 256 | 0.7508 | 0.0461 | 0.9922 | 0.8633 |
| b100 | 100 | full_source | 2 | 256 | 0.7466 | 0.038 | 1 | 1 |
| b100 | 100 | full_source_shuffle | 2 | 154 | 0.7588 | 0.0391 | 1 | 1 |
| b100 | 100 | height_scalar | 1 | 256 | 0.0045 | 0.0008 | 0.0039 | 0.0039 |
| b100 | 100 | output_close_open_scalar | 2 | 256 | 0.7466 | 0.0487 | 1 | 0.7773 |

![axis_span_bracket_ci](../../figures/dyck_counter_task_a_axis_span_patching/axis_span_bracket_ci.png)

![axis_span_full_vocab_ci](../../figures/dyck_counter_task_a_axis_span_patching/axis_span_full_vocab_ci.png)

![axis_span_best_layer_heatmap](../../figures/dyck_counter_task_a_axis_span_patching/axis_span_best_layer_heatmap.png)

### Sparse Sweep: Accuracy/CE 指标定义

这里不再只看一个 accuracy，因为 sparse setting 里有两个不同问题会混在一起：模型是否知道下一步该 open/close，以及模型是否把足够概率分给 bracket token 而不是 noise token。

- **Dyck-target**：只在下一 token 是 `open` 或 `close` 的位置上评估；不包括普通 noise 位置。
- **forced/free split**：`forced` 是 Dyck 规则唯一决定下一 bracket 的位置；`free` 是 open/close 都合法、generator 随机采样的位置。`forced` 更适合检验模型是否学会 Dyck 约束；`free` 的上限接近随机猜测。
- **full-vocab acc / full-vocab NLL**：在完整 vocabulary 上评估正确 bracket token。它同时受 open/close 判断和 bracket-vs-noise 竞争影响。
- **bracket-only acc / bracket-only NLL**：只在 `open` 和 `close` 两个 logits 上重新 softmax。它隔离了 open/close 方向是否正确，不考察 bracket 是否赢过 noise。
- **bracket mass**：完整 softmax 下 `P(open) + P(close)`。它直接衡量模型有没有把概率质量放到 bracket token 上。
- **train eval loss**：训练 pipeline 的全 token loss。因为 `seq_len=2000` 里多数位置是 noise，它对 Dyck 行为变化不敏感。

### Sparse Sweep: 关键结果

**读法先定清楚。** `full-vocab` 指标回答“模型最终是否真的输出正确 bracket token”；`bracket-only` 指标回答“如果只在 open/close 里二选一，方向是否对”；`bracket mass` 回答“模型给 open+close 的总概率够不够”。

**主要结果。** b20 的 all-Dyck full-vocab acc 只有 0.098，但 bracket-only acc 是 0.674，bracket mass 只有 0.015。这说明 b20 不是完全不会区分 open/close，而是 bracket token 在完整 vocab 里概率质量太低。

**forced split 更能看出 Dyck 约束是否学会。** b20 forced bracket-only acc 已经是 0.989，但 forced full-vocab acc 只有 0.263。到 b48 forced full-vocab acc 升到 0.826，b100 到 0.988。这说明 sparse transition 主要发生在 bracket token 能否赢过 noise，而不是 forced open/close 规则本身。

**free split 不应被解读成必须高 accuracy。** free step 的 open/close 由随机采样决定，bracket-only acc 接近 0.5 是合理的。b100 free bracket-only acc 是 0.503，full-vocab acc 是 0.503，接近随机 ceiling。

**CE 比 accuracy 更连续，但要看 Dyck-target CE，不是全 token loss。** all-Dyck full-vocab NLL 从 b20 的 4.798 降到 b100 的 3.597，再到 b400 的 2.242；但 train eval loss 在 b20/b100 几乎不变 (4.178 vs 4.180)。所以全 token loss 被 noise positions 稀释，不能替代 Dyck-target metrics。

Dyck-target metrics, selected densities:

| setting | bracket_tokens | full_vocab_acc | bracket_acc | full_vocab_nll | bracket_nll | bracket_mass | forced_full_acc | train_eval_loss |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| b20 | 20 | 0.0983 | 0.6744 | 4.798 | 0.4906 | 0.015 | 0.2634 | 4.178 |
| b40 | 40 | 0.2263 | 0.621 | 4.382 | 0.558 | 0.0227 | 0.7772 | 4.184 |
| b48 | 48 | 0.227 | 0.6077 | 4.267 | 0.5814 | 0.0261 | 0.8255 | 4.186 |
| b64 | 64 | 0.566 | 0.6011 | 3.959 | 0.5908 | 0.0349 | 0.9773 | 4.186 |
| b100 | 100 | 0.5782 | 0.5787 | 3.597 | 0.6156 | 0.0513 | 0.9877 | 4.18 |
| b400 | 400 | 0.5399 | 0.5399 | 2.242 | 0.6594 | 0.2062 | 0.9868 | 3.959 |

Forced/free split, selected densities:

| setting | bracket_tokens | forced_full_acc | forced_bracket_acc | forced_full_nll | forced_bracket_nll | free_full_acc | free_bracket_acc | free_full_nll | free_bracket_nll |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| b20 | 20 | 0.2634 | 0.9889 | 4.312 | 0.0717 | 0.0055 | 0.4978 | 5.072 | 0.726 |
| b40 | 40 | 0.7772 | 0.9943 | 3.856 | 0.0893 | 0.043 | 0.4968 | 4.557 | 0.7139 |
| b48 | 48 | 0.8255 | 0.9813 | 3.734 | 0.1427 | 0.0541 | 0.4998 | 4.421 | 0.7081 |
| b64 | 64 | 0.9773 | 0.9942 | 3.534 | 0.1404 | 0.4629 | 0.5026 | 4.065 | 0.7037 |
| b100 | 100 | 0.9877 | 0.9906 | 3.164 | 0.1486 | 0.5026 | 0.5027 | 3.677 | 0.7018 |
| b400 | 400 | 0.9868 | 0.9868 | 1.722 | 0.2243 | 0.5017 | 0.5017 | 2.287 | 0.6967 |

![sparse_loss_behavior](../../figures/dyck_counter_task_a_sparse_geometry/sparse_loss_behavior.png)

### Direction Geometry and Final-hidden Ablation

这里把两个问题分开：

1. **direction geometry**：看 ridge count directions、当前 bracket 方向、forced-next 方向，与 input/output bracket vectors 的夹角。`output_close_open = W_close - W_open`，`output_bracket_noise = mean(W_close,W_open)-mean(W_noise)`。

2. **final-hidden ablation**：在最终 hidden state 上移除某个方向/子空间的 centered projection，再用同一个 output head 解码。这不是完整 online activation patch；它只回答“这个方向是否被 final output readout 因果使用”。因此它能和上面的 online patching 互相校验。

Final-layer key alignments, selected densities:

| setting | bracket_tokens | layer | vector_a | vector_b | cosine | abs_cosine | angle_deg |
| --- | --- | --- | --- | --- | --- | --- | --- |
| b20 | 20 | 2 | current_bracket | input_close_open | 0.5263 | 0.5263 | 58.24 |
| b20 | 20 | 2 | current_bracket | output_bracket_noise | -0.1206 | 0.1206 | 96.92 |
| b20 | 20 | 2 | forced_next | height | 0.0576 | 0.0576 | 86.7 |
| b20 | 20 | 2 | forced_next | output_close_open | 0.9477 | 0.9477 | 18.61 |
| b20 | 20 | 2 | height | input_close_open | 0.0293 | 0.0293 | 88.32 |
| b20 | 20 | 2 | height | output_close_open | -0.0207 | 0.0207 | 91.18 |
| b40 | 40 | 2 | current_bracket | input_close_open | 0.5844 | 0.5844 | 54.24 |
| b40 | 40 | 2 | current_bracket | output_bracket_noise | -0.1193 | 0.1193 | 96.85 |
| b40 | 40 | 2 | forced_next | height | 0.0523 | 0.0523 | 87 |
| b40 | 40 | 2 | forced_next | output_close_open | 0.933 | 0.933 | 21.09 |
| b40 | 40 | 2 | height | input_close_open | -0.0163 | 0.0163 | 90.93 |
| b40 | 40 | 2 | height | output_close_open | -0.0159 | 0.0159 | 90.91 |
| b48 | 48 | 2 | current_bracket | input_close_open | 0.5516 | 0.5516 | 56.52 |
| b48 | 48 | 2 | current_bracket | output_bracket_noise | -0.0499 | 0.0499 | 92.86 |
| b48 | 48 | 2 | forced_next | height | 0.0545 | 0.0545 | 86.88 |
| b48 | 48 | 2 | forced_next | output_close_open | 0.9325 | 0.9325 | 21.17 |
| b48 | 48 | 2 | height | input_close_open | 0.1406 | 0.1406 | 81.92 |
| b48 | 48 | 2 | height | output_close_open | -0.0254 | 0.0254 | 91.45 |
| b64 | 64 | 2 | current_bracket | input_close_open | 0.5509 | 0.5509 | 56.57 |
| b64 | 64 | 2 | current_bracket | output_bracket_noise | -0.1781 | 0.1781 | 100.3 |
| b64 | 64 | 2 | forced_next | height | 0.0524 | 0.0524 | 87 |
| b64 | 64 | 2 | forced_next | output_close_open | 0.961 | 0.961 | 16.06 |
| b64 | 64 | 2 | height | input_close_open | 0.0093 | 0.0093 | 89.46 |
| b64 | 64 | 2 | height | output_close_open | 0.0184 | 0.0184 | 88.94 |
| b100 | 100 | 2 | current_bracket | input_close_open | 0.5326 | 0.5326 | 57.82 |
| b100 | 100 | 2 | current_bracket | output_bracket_noise | -0.0814 | 0.0814 | 94.67 |
| b100 | 100 | 2 | forced_next | height | 0.0405 | 0.0405 | 87.68 |
| b100 | 100 | 2 | forced_next | output_close_open | 0.948 | 0.948 | 18.56 |
| b100 | 100 | 2 | height | input_close_open | -0.0345 | 0.0345 | 91.98 |
| b100 | 100 | 2 | height | output_close_open | -0.0315 | 0.0315 | 91.81 |
| b400 | 400 | 2 | current_bracket | input_close_open | 0.5872 | 0.5872 | 54.04 |
| b400 | 400 | 2 | current_bracket | output_bracket_noise | -0.0961 | 0.0961 | 95.51 |
| b400 | 400 | 2 | forced_next | height | 0.023 | 0.023 | 88.68 |
| b400 | 400 | 2 | forced_next | output_close_open | 0.9091 | 0.9091 | 24.62 |
| b400 | 400 | 2 | height | input_close_open | 0.0593 | 0.0593 | 86.6 |
| b400 | 400 | 2 | height | output_close_open | -0.0061 | 0.0061 | 90.35 |

Forced-split final-hidden direction ablation, selected densities:

| setting | bracket_tokens | ablation | delta_full_vocab_acc_vs_baseline | delta_bracket_acc_vs_baseline | delta_full_vocab_nll_vs_baseline | delta_bracket_nll_vs_baseline | delta_bracket_mass_vs_baseline |
| --- | --- | --- | --- | --- | --- | --- | --- |
| b20 | 20 | remove_forced_next | -0.262 | -0.5268 | 2.031 | 0.6294 | -0.0134 |
| b20 | 20 | remove_height | 0.0019 | 0 | -0.0037 | -0.0004 | 0 |
| b20 | 20 | remove_output_bracket_noise | 0.4566 | -0.0003 | -0.8138 | 0.0011 | 0.0314 |
| b20 | 20 | remove_output_close_open | -0.2634 | -0.4884 | 2.068 | 0.6292 | -0.0134 |
| b20 | 20 | remove_random | -0.0003 | 0 | 0 | 0 | -0 |
| b40 | 40 | remove_forced_next | -0.7709 | -0.4726 | 1.606 | 0.6123 | -0.0154 |
| b40 | 40 | remove_height | 0.0033 | 0 | -0.0029 | -0.0002 | 0.0001 |
| b40 | 40 | remove_output_bracket_noise | 0.1214 | -0.0004 | -0.653 | 0.0016 | 0.0281 |
| b40 | 40 | remove_output_close_open | -0.7753 | -0.4873 | 1.633 | 0.6091 | -0.0154 |
| b40 | 40 | remove_random | -0.0857 | -0.0004 | 0.0673 | 0.0066 | -0.0017 |
| b48 | 48 | remove_forced_next | -0.8232 | -0.5162 | 1.469 | 0.5596 | -0.0181 |
| b48 | 48 | remove_height | 0.0022 | 0.0007 | -0.0008 | -0.0007 | 0 |
| b48 | 48 | remove_output_bracket_noise | 0.0434 | -0.0002 | -0.5652 | 0.0015 | 0.0288 |
| b48 | 48 | remove_output_close_open | -0.8228 | -0.5067 | 1.469 | 0.5522 | -0.0182 |
| b48 | 48 | remove_random | -0.0162 | -0.0002 | 0.0242 | 0.0024 | -0.0007 |
| b64 | 64 | remove_forced_next | -0.938 | -0.5453 | 1.232 | 0.5615 | -0.0154 |
| b64 | 64 | remove_height | 0.0002 | -0.0002 | 0.0011 | 0.0003 | -0 |
| b64 | 64 | remove_output_bracket_noise | 0.0133 | -0.0002 | -0.5379 | 0.0009 | 0.0305 |
| b64 | 64 | remove_output_close_open | -0.9028 | -0.5027 | 1.237 | 0.5538 | -0.0155 |
| b64 | 64 | remove_random | 0.0002 | -0.0003 | 0.0078 | 0.0019 | -0.0003 |
| b100 | 100 | remove_forced_next | -0.8561 | -0.5913 | 1.298 | 0.5543 | -0.0234 |
| b100 | 100 | remove_height | 0 | 0 | -0.0003 | -0.0005 | -0 |
| b100 | 100 | remove_output_bracket_noise | 0.003 | 0.0001 | -0.6688 | -0.0004 | 0.0644 |
| b100 | 100 | remove_output_close_open | -0.8937 | -0.4851 | 1.366 | 0.545 | -0.0245 |
| b100 | 100 | remove_random | -0.0001 | 0 | 0.0375 | 0.0052 | -0.0017 |
| b400 | 400 | remove_forced_next | -0.6456 | -0.6048 | 1.124 | 0.4789 | -0.0857 |
| b400 | 400 | remove_height | 0 | 0 | 0.0017 | -0.0001 | -0.0003 |
| b400 | 400 | remove_output_bracket_noise | 0.0003 | 0.0003 | -0.3927 | -0.0001 | 0.1434 |
| b400 | 400 | remove_output_close_open | -0.4943 | -0.4698 | 1.187 | 0.4684 | -0.0939 |
| b400 | 400 | remove_random | 0.0002 | 0.0002 | -0.0005 | 0 | -0 |

![sparse_direction_alignment](../../figures/dyck_counter_task_a_sparse_geometry/sparse_direction_alignment.png)

![sparse_direction_stability](../../figures/dyck_counter_task_a_sparse_geometry/sparse_direction_stability.png)

![sparse_direction_ablation](../../figures/dyck_counter_task_a_sparse_geometry/sparse_direction_ablation.png)

### 结果解释

**1. Count probe 可读，不等于模型沿 count direction 输出。**

axis/span patching 直接比较“只 patch 某条方向”是否能翻转 forced open/close。结果很明确：`output_close_open` 和 `forced_next` 方向有效，`height` 方向基本无效。b20 的 bracket CI：output_close_open=0.869，forced_next=0.872，height=0.028；b100 对应为 0.747、0.751、0.004。所以 hidden state 中虽然能线性读出 height/left/right，但最终 bracket decision 主要接在另一个 readout-aligned direction 上。`full_source` 和 `full_source_shuffle` 很接近，也说明 full-state patch 的强效应不能解释成逐样本 source counter state 被精确移植。

**2. sparse setting 的失败主要分成两个子问题：open/close 是否对，以及 bracket 是否赢过 noise。**

b20 的 all-Dyck full-vocab acc 很低：0.098；但 bracket-only acc 是 0.674，说明 open/close 子空间不是完全失效。真正薄弱的是 bracket mass：b20=0.015，b48=0.026，b100=0.051。因此 full-vocab accuracy 低，混合了两件事：模型是否知道 open/close，以及 bracket token 是否能在完整 vocab 中胜过 noise。后者是低 density 下的主要瓶颈。

**3. 全 token eval loss 不适合判断 Dyck 行为是否学会。**

b20 和 b100 的 train eval loss 几乎一样：4.178 vs 4.180；但 forced full-vocab acc 从 0.263 升到 0.988。原因是 seq_len=2000 中绝大多数位置是 noise，全 token loss 被 noise prediction 主导。因此这里更应该报告 Dyck-target CE/acc、forced/free split、bracket-only 指标和 bracket mass。

**4. Geometry 和 ablation 支持同一个机制解释。**

final layer 中，`forced_next` 与 `output_close_open` 高度对齐：abs cosine b20=0.948，b48=0.932。但 `height` 与 `output_close_open` 基本正交：b20=0.021。这说明输出头读到的是 forced-decision/readout axis，不是 ridge height axis。final-hidden ablation 也一致：在 forced split 上移除 `output_close_open` 显著增加 bracket NLL (b20 +0.629, b100 +0.545)；移除 `height` 的影响接近 0 (b20 -0.000, b100 -0.001)。

**结论。** 当前最稳妥的说法是：模型确实表示了 count/left/right，但 next-bracket 输出不是直接沿 height ridge direction 完成的；真正接到 output head 的是与 `output_close_open` 对齐的 forced-decision direction。低 density 的主要失败来自 full-vocab bracket-vs-noise 竞争，以及这个 forced-decision readout 是否稳定形成。

In [ ]:
# Re-run this block if needed
# !conda run -n hse python scripts/task_a_axis_span_patching.py --settings b20 b48 b100 --pairs-per-setting 256 --batch-size 8 --max-batches 260 --device cuda
# !conda run -n hse python scripts/task_a_sparse_geometry_diagnostics.py --max-rows 160000
# !conda run -n hse python scripts/append_task_a_extended_patching_geometry.py